In [ ]:
!pip install pandas numpy tqdm torch sentence-transformers rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 40.4 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import re
from collections import Counter
from tqdm import tqdm

import torch
from sentence_transformers import SentenceTransformer
from rapidfuzz.distance import Levenshtein

USE_GPU = torch.cuda.is_available()
DEVICE = "cuda" if USE_GPU else "cpu"


words_df = pd.read_csv("unique_words.csv")
words = words_df["word"].dropna().unique().tolist()


def normalize(word):
    return re.sub(r"[^\u0900-\u097F]", "", word.strip())


words = [normalize(w) for w in words if w.strip() != ""]


freq = Counter(words)

valid_seed = set([
    "है", "था", "और", "मैं", "तुम", "वह", "यह",
    "कर", "रहा", "गया", "किया", "लोग", "समय"
])


model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2", device=DEVICE)


batch_size = 512

def compute_embeddings(word_list):
    embeddings = []
    for i in tqdm(range(0, len(word_list), batch_size)):
        batch = word_list[i:i+batch_size]
        emb = model.encode(batch, convert_to_tensor=True, device=DEVICE)
        embeddings.append(emb)
    return torch.cat(embeddings)


word_embeddings = compute_embeddings(words)


def classify_word(idx, word):
    f = freq[word]

    if f > 50:
        return "correct", "high", "high frequency"

    if word in valid_seed:
        return "correct", "high", "seed lexicon"

    min_dist = float("inf")
    closest = None

    for w in valid_seed:
        d = Levenshtein.distance(word, w)
        if d < min_dist:
            min_dist = d
            closest = w

    if min_dist <= 1:
        return "incorrect", "high", f"edit distance to {closest}"

    emb = word_embeddings[idx]
    sims = torch.matmul(word_embeddings, emb)

    top_sim = torch.max(sims).item()

    if top_sim > 0.85:
        return "correct", "medium", "semantic similarity"

    return "incorrect", "low", "uncertain"


results = []

for i, word in enumerate(tqdm(words)):
    label, confidence, reason = classify_word(i, word)

    results.append({
        "word": word,
        "label": label,
        "confidence": confidence,
        "reason": reason
    })


out_df = pd.DataFrame(results)


out_df.to_csv("q3_word_classification.csv", index=False)


print(out_df["label"].value_counts())
print(out_df["confidence"].value_counts())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

100%|██████████| 177421/177421 [05:50<00:00, 506.03it/s]


label
correct      175489
incorrect      1932
Name: count, dtype: int64
confidence
medium    174784
high        2637
Name: count, dtype: int64


In [ ]:
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from rapidfuzz.distance import Levenshtein
from tqdm import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

df = pd.read_csv("/content/q4_task.csv")  


model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2", device=DEVICE)


def tokenize(text):
    return str(text).strip().split()


def wer(ref, hyp):
    ref, hyp = tokenize(ref), tokenize(hyp)
    dp = np.zeros((len(ref)+1, len(hyp)+1))

    for i in range(len(ref)+1):
        dp[i][0] = i
    for j in range(len(hyp)+1):
        dp[0][j] = j

    for i in range(1, len(ref)+1):
        for j in range(1, len(hyp)+1):
            cost = 0 if ref[i-1] == hyp[j-1] else 1
            dp[i][j] = min(
                dp[i-1][j] + 1,
                dp[i][j-1] + 1,
                dp[i-1][j-1] + cost
            )

    return dp[len(ref)][len(hyp)] / len(ref)


def align(ref, hyp):
    ref, hyp = tokenize(ref), tokenize(hyp)
    n, m = len(ref), len(hyp)

    dp = np.zeros((n+1, m+1))

    for i in range(n+1):
        dp[i][0] = i
    for j in range(m+1):
        dp[0][j] = j

    for i in range(1, n+1):
        for j in range(1, m+1):
            cost = 0 if ref[i-1] == hyp[j-1] else 1
            dp[i][j] = min(
                dp[i-1][j] + 1,
                dp[i][j-1] + 1,
                dp[i-1][j-1] + cost
            )

    i, j = n, m
    aligned = []

    while i > 0 and j > 0:
        if ref[i-1] == hyp[j-1]:
            aligned.append((ref[i-1], hyp[j-1]))
            i -= 1
            j -= 1
        else:
            if dp[i][j] == dp[i-1][j-1] + 1:
                aligned.append((ref[i-1], hyp[j-1]))
                i -= 1
                j -= 1
            elif dp[i][j] == dp[i-1][j] + 1:
                aligned.append((ref[i-1], None))
                i -= 1
            else:
                aligned.append((None, hyp[j-1]))
                j -= 1

    aligned.reverse()
    return aligned


def build_lattice(reference, models):
    ref_tokens = tokenize(reference)
    lattice = [[tok] for tok in ref_tokens]

    for m in models:
        aligned = align(reference, m)
        idx = 0

        for r, h in aligned:
            if r is not None:
                if h and h not in lattice[idx]:
                    lattice[idx].append(h)
                idx += 1

    return lattice


def refine_lattice(lattice, models):
    tokenized_models = [tokenize(m) for m in models]

    for i in range(len(lattice)):
        votes = {}

        for tokens in tokenized_models:
            if i < len(tokens):
                w = tokens[i]
                votes[w] = votes.get(w, 0) + 1

        for w, c in votes.items():
            if c >= 3 and w not in lattice[i]:
                lattice[i].append(w)

    return lattice


def semantic_match(word, candidates):
    if word in candidates:
        return True

    emb = model.encode([word] + candidates, convert_to_tensor=True, device=DEVICE)
    sims = torch.matmul(emb[0], emb[1:].T)

    return torch.max(sims).item() > 0.8


def lattice_wer(reference, hyp, lattice):
    hyp_tokens = tokenize(hyp)

    errors = 0
    for i in range(min(len(hyp_tokens), len(lattice))):
        if not semantic_match(hyp_tokens[i], lattice[i]):
            errors += 1

    errors += abs(len(hyp_tokens) - len(lattice))

    return errors / len(lattice)


results = []

model_cols = ["Model H", "Model i", "Model k", "Model l", "Model m", "Model n"]

for _, row in tqdm(df.iterrows(), total=len(df)):
    ref = row["Human"]
    models = [row[col] for col in model_cols]

    lattice = build_lattice(ref, models)
    lattice = refine_lattice(lattice, models)

    for i, col in enumerate(model_cols):
        m = models[i]

        standard = wer(ref, m)
        lattice_score = lattice_wer(ref, m, lattice)

        results.append({
            "model": col,
            "standard_wer": standard,
            "lattice_wer": lattice_score
        })


out_df = pd.DataFrame(results)

# -------- AGGREGATE --------
summary = out_df.groupby("model").mean().reset_index()

summary.to_csv("q4_results.csv", index=False)
print(summary)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
100%|██████████| 46/46 [00:06<00:00,  7.43it/s]

     model  standard_wer  lattice_wer
0  Model H      0.039773     0.013640
1  Model i      0.073036     0.003623
2  Model k      0.244265     0.034622
3  Model l      0.113239     0.024015
4  Model m      0.207301     0.027274
5  Model n      0.113881     0.027861
